<div align="center">

# 🧠 Lab Infra — End-to-End Demo

**RAG API · Hybrid LangGraph Agent · RAGAS Evaluation · Model Benchmark**

</div>

---

This notebook demonstrates the full AI pipeline running on the bare-metal Docker Swarm lab:

| Section | What it shows |
|---------|---------------|
| **1. Setup** | Dependencies + service URLs |
| **2. RAG — Ingest** | Upload a document → chunk → embed → store in Qdrant |
| **3. RAG — Query** | Semantic search + LLM answer with sources |
| **4. Agent — RAG route** | Question routed to RAG tool by LangGraph |
| **5. Agent — Data route** | Question routed to SQL tool (Postgres) by LangGraph |
| **6. RAGAS metrics** | Faithfulness · Answer Relevancy · Context Precision |
| **7. Model Benchmark** | Multi-model leaderboard by category |
| **8. Agent traces** | Latency distribution · Route breakdown from OpenSearch |

> **All inference is local** — Ollama on RTX 2080 Ti (master2). No cloud, no API keys.

## 1. Setup

In [ ]:
# Install dependencies not present in the llm kernel by default
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "httpx", "boto3", "opensearch-py", "matplotlib", "pandas", "numpy"],
               check=True)
print("✅ Dependencies ready")

In [ ]:
import httpx
import boto3
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime, timedelta

# ── Service URLs (internal overlay DNS — only valid inside the cluster) ─────
RAG_API_URL    = "http://rag-api:8000"       # overlay: rag-api_rag-api service
AGENT_URL      = "http://agent_agent:8000"   # overlay: agent_agent service
OPENSEARCH_URL = "http://opensearch:9200"     # internal overlay DNS

# ── MinIO / S3 config ────────────────────────────────────────────────────────
MINIO_ENDPOINT   = "http://minio_minio:9000"  # overlay: minio_minio service
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "minioadmin")
GOVERNANCE_BUCKET = "governance"

# ── Qdrant collection ────────────────────────────────────────────────────────
COLLECTION = "lab_documents_bge"  # 1024d cosine, bge-m3

# ── httpx client (plain HTTP for overlay, no TLS needed inside cluster) ─────
client = httpx.Client(verify=False, timeout=120.0)

print("✅ Config loaded")
print(f"  RAG API : {RAG_API_URL}")
print(f"  Agent   : {AGENT_URL}")
print(f"  MinIO   : {MINIO_ENDPOINT}")
print(f"  OpenSearch: {OPENSEARCH_URL}")

In [ ]:
# ── Health check — both services must be up before running the demo ──────────
for name, url in [("RAG API", f"{RAG_API_URL}/health"), ("Agent", f"{AGENT_URL}/health")]:
    r = client.get(url)
    r.raise_for_status()
    body = r.json()
    status = body.get("status", "ok")
    print(f"  {'✅' if status == 'ok' else '⚠️'} {name}: {status}")
    if name == "RAG API":
        for svc, info in body.get("backends", {}).items():
            print(f"      └─ {svc}: {info.get('status')}")

## 2. RAG — Ingest Document

We upload `notebooks/data/intro_llm.txt` — a short introduction to LLMs.

The RAG API pipeline:
1. Stores the raw file in **MinIO** (`rag-documents` bucket)
2. Splits text into chunks (`RecursiveCharacterTextSplitter`)
3. Embeds each chunk via **Ollama** (`bge-m3`, 1024d)
4. Upserts vectors into **Qdrant** (`lab_documents_bge`)
5. Stores metadata + chunk text in **PostgreSQL** (`embeddings` table)

In [ ]:
doc_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data", "intro_llm.txt")

with open(doc_path, "rb") as f:
    response = client.post(
        f"{RAG_API_URL}/ingest/",
        files={"file": ("intro_llm.txt", f, "text/plain")},
        data={"collection": COLLECTION},
    )

response.raise_for_status()
ingest_result = response.json()

print("✅ Document ingested successfully")
print(f"  document_id   : {ingest_result['document_id']}")
print(f"  chunks_indexed: {ingest_result['chunks_indexed']}")
print(f"  minio_path    : {ingest_result['minio_path']}")
print(f"  collection    : {ingest_result['collection']}")

## 3. RAG — Query

Three questions of increasing complexity. Each answer is grounded in the ingested document chunks.

The pipeline: question → `bge-m3` embed → Qdrant ANN search → top-K chunks → Ollama LLM → answer + sources.

In [ ]:
questions = [
    "What is RAG and how does it reduce hallucinations?",
    "What RAGAS metrics are used to evaluate RAG quality?",
    "What is the difference between nomic-embed-text and bge-m3 embeddings?",
]

rag_results = []

for q in questions:
    r = client.post(
        f"{RAG_API_URL}/query/",
        json={"question": q, "collection": COLLECTION, "top_k": 3},
    )
    r.raise_for_status()
    result = r.json()
    rag_results.append(result)

    print(f"\n{'─'*70}")
    print(f"❓ {q}")
    print(f"💬 {result['answer'][:300]}{'...' if len(result['answer']) > 300 else ''}")
    print(f"📚 Sources ({len(result['sources'])}):",
          ", ".join(f"{s['filename']} (score={s['score']:.3f})" for s in result['sources']))

## 4. Hybrid Agent — RAG Route

The LangGraph agent decides HOW to answer:
- **RAG route** → semantic search in Qdrant
- **Data route** → SQL query against PostgreSQL
- **Both** → parallel execution + synthesis

Questions about documents → RAG. Questions about data → SQL.

In [ ]:
agent_q_rag = "Explain how the attention mechanism works in Large Language Models"

r = client.post(
    f"{AGENT_URL}/agent/query",
    json={"question": agent_q_rag, "session_id": "demo-notebook-rag"},
)
r.raise_for_status()
agent_rag_result = r.json()

print("🤖 HYBRID AGENT — RAG Route")
print(f"   route      : {agent_rag_result['route']}")
print(f"   tool_used  : {agent_rag_result['tool_used']}")
print(f"   latency_ms : {agent_rag_result['latency_ms']:.0f} ms")
print(f"   chunks     : {agent_rag_result['chunks_retrieved']}")
print(f"   model      : {agent_rag_result['model']}")
print(f"\n💬 Answer:\n{agent_rag_result['answer'][:500]}")
if agent_rag_result.get('sources'):
    print(f"\n📚 Sources: {[s['filename'] for s in agent_rag_result['sources']]}")

## 5. Hybrid Agent — Data Route

A question about structured data triggers the **SQL tool**: `qwen2.5-coder:7b` generates a SQL query, executes it against PostgreSQL, and the synthesizer formats the answer.

In [ ]:
agent_q_data = "How many documents have been ingested in the RAG system?"

r = client.post(
    f"{AGENT_URL}/agent/query",
    json={"question": agent_q_data, "session_id": "demo-notebook-data"},
)
r.raise_for_status()
agent_data_result = r.json()

print("🤖 HYBRID AGENT — Data Route")
print(f"   route      : {agent_data_result['route']}")
print(f"   tool_used  : {agent_data_result['tool_used']}")
print(f"   latency_ms : {agent_data_result['latency_ms']:.0f} ms")
print(f"   model      : {agent_data_result['model']}")
if agent_data_result.get('sql_query'):
    print(f"\n🗄️  SQL generated:\n   {agent_data_result['sql_query']}")
print(f"\n💬 Answer:\n{agent_data_result['answer'][:500]}")

## 6. RAGAS Evaluation Metrics

Every Sunday at 04:00, an **Airflow DAG** (`agent_ragas_eval`) runs LLM-as-judge evaluation on a synthetic Q&A dataset and stores the results in MinIO.

We read the latest results and visualize the three core metrics:

| Metric | What it measures | Target |
|--------|-----------------|--------|
| **Faithfulness** | Are claims in the answer supported by the retrieved context? | > 0.8 |
| **Answer Relevancy** | Is the answer on-topic with respect to the question? | > 0.8 |
| **Context Precision** | Are the retrieved chunks actually useful for answering? | > 0.7 |

In [ ]:
# ── Connect to MinIO (S3-compatible) ─────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    verify=False,
)

# ── Find the latest RAGAS result ──────────────────────────────────────────────
paginator = s3.get_paginator("list_objects_v2")
ragas_files = []
for page in paginator.paginate(Bucket=GOVERNANCE_BUCKET, Prefix="ragas-results/"):
    ragas_files.extend(page.get("Contents", []))

ragas_files = [f for f in ragas_files if f["Key"].endswith("results.json")]
ragas_files.sort(key=lambda x: x["LastModified"], reverse=True)

if not ragas_files:
    print("⚠️  No RAGAS results found in MinIO yet.")
    print("   Run the 'agent_ragas_eval' Airflow DAG to generate them.")
else:
    latest_key = ragas_files[0]["Key"]
    print(f"✅ Latest RAGAS result: {latest_key}")
    obj = s3.get_object(Bucket=GOVERNANCE_BUCKET, Key=latest_key)
    ragas_data = json.loads(obj["Body"].read())
    print(f"   run_date      : {ragas_data.get('run_date')}")
    print(f"   judge_model   : {ragas_data.get('judge_model')}")
    print(f"   total_records : {ragas_data.get('total_records')}")
    agg = ragas_data.get("aggregate", {})
    for metric, vals in agg.items():
        mean = vals.get('mean', 0)
        bar = '█' * int(mean * 20) + '░' * (20 - int(mean * 20))
        status = '✅' if mean >= 0.7 else '⚠️'
        print(f"   {status} {metric:<22} {bar} {mean:.3f}")

In [ ]:
# ── Visualize RAGAS metrics ───────────────────────────────────────────────────
if ragas_files and 'ragas_data' in dir():
    agg = ragas_data.get("aggregate", {})
    metrics = list(agg.keys())
    means   = [agg[m]["mean"] for m in metrics]
    mins    = [agg[m]["min"]  for m in metrics]
    maxs    = [agg[m]["max"]  for m in metrics]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"RAGAS Evaluation — {ragas_data.get('run_date')} "
                 f"(judge: {ragas_data.get('judge_model')})",
                 fontsize=13, fontweight="bold")

    # ── Bar chart: mean + range ───────────────────────────────────────────────
    ax = axes[0]
    colors = ["#2ecc71" if m >= 0.8 else "#f39c12" if m >= 0.7 else "#e74c3c" for m in means]
    bars = ax.barh(metrics, means, color=colors, height=0.5, zorder=2)
    for i, (mn, mx, mean) in enumerate(zip(mins, maxs, means)):
        ax.plot([mn, mx], [i, i], color="gray", linewidth=2, zorder=3)
        ax.text(mean + 0.01, i, f"{mean:.3f}", va="center", fontsize=10)
    ax.axvline(0.8, color="green", linestyle="--", alpha=0.5, label="target (0.8)")
    ax.axvline(0.7, color="orange", linestyle="--", alpha=0.5, label="minimum (0.7)")
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("Score")
    ax.set_title("Mean Score (bar = min–max range)")
    ax.legend(fontsize=9)
    ax.grid(axis="x", alpha=0.3, zorder=1)
    ax.set_facecolor("#f9f9f9")

    # ── Per-record scatter ────────────────────────────────────────────────────
    ax2 = axes[1]
    records = ragas_data.get("records", [])
    if records:
        for i, metric in enumerate(metrics):
            scores = [r["scores"].get(metric, 0) for r in records]
            jitter = np.random.uniform(-0.15, 0.15, len(scores))
            ax2.scatter(scores, [i] * len(scores) + jitter,
                        alpha=0.6, s=60, color=colors[i], zorder=2)
    ax2.axvline(0.8, color="green", linestyle="--", alpha=0.5)
    ax2.axvline(0.7, color="orange", linestyle="--", alpha=0.5)
    ax2.set_xlim(0, 1.05)
    ax2.set_yticks(range(len(metrics)))
    ax2.set_yticklabels(metrics)
    ax2.set_xlabel("Score per record")
    ax2.set_title("Per-record Distribution")
    ax2.grid(axis="x", alpha=0.3, zorder=1)
    ax2.set_facecolor("#f9f9f9")

    plt.tight_layout()
    plt.savefig("ragas_metrics.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("📊 Chart saved to ragas_metrics.png")

## 7. Model Benchmark — Leaderboard

Every Sunday at 06:00, an **Airflow DAG** (`agent_model_benchmark`) evaluates all available Ollama models across three categories:

- **Instruction following** — Does the model follow explicit instructions?
- **Reasoning** — Multi-step logical reasoning tasks
- **Coding** — Code generation and debugging

Results are stored in MinIO and indexed in OpenSearch for trending.

In [ ]:
# ── Load latest benchmark results ─────────────────────────────────────────────
bench_files = []
for page in paginator.paginate(Bucket=GOVERNANCE_BUCKET, Prefix="benchmarks/"):
    bench_files.extend(page.get("Contents", []))

bench_files = [f for f in bench_files if f["Key"].endswith("results.json")]
bench_files.sort(key=lambda x: x["LastModified"], reverse=True)

if not bench_files:
    print("⚠️  No benchmark results found in MinIO yet.")
    print("   Run the 'agent_model_benchmark' Airflow DAG to generate them.")
else:
    latest_bench_key = bench_files[0]["Key"]
    print(f"✅ Latest benchmark: {latest_bench_key}")
    obj = s3.get_object(Bucket=GOVERNANCE_BUCKET, Key=latest_bench_key)
    bench_data = json.loads(obj["Body"].read())
    print(f"   run_date      : {bench_data.get('run_date')}")
    print(f"   models_tested : {bench_data.get('models_tested')}")
    print(f"   total_questions: {bench_data.get('total_questions')}")
    print()
    print(f"{'Rank':<5} {'Model':<30} {'Overall':>8} {'Instruct':>9} {'Reason':>8} {'Coding':>8}")
    print("─" * 72)
    for i, entry in enumerate(bench_data.get("leaderboard", []), 1):
        model = entry.get("model", "?")
        overall = entry.get("overall_score", 0)
        cats = entry.get("scores_by_category", {})
        inst = cats.get("instruction_following", 0)
        reas = cats.get("reasoning", 0)
        code = cats.get("coding", 0)
        medal = ["🥇", "🥈", "🥉"].get(i - 1, f"#{i}")
        print(f"{medal:<5} {model:<30} {overall:>8.3f} {inst:>9.3f} {reas:>8.3f} {code:>8.3f}")

In [ ]:
# ── Visualize benchmark leaderboard ──────────────────────────────────────────
if bench_files and 'bench_data' in dir():
    leaderboard = bench_data.get("leaderboard", [])
    if not leaderboard:
        print("No leaderboard data available")
    else:
        models     = [e["model"].split(":")[0] for e in leaderboard]  # strip :tag
        categories = ["instruction_following", "reasoning", "coding"]
        cat_labels = ["Instruction Following", "Reasoning", "Coding"]
        cat_colors = ["#3498db", "#9b59b6", "#e67e22"]

        x      = np.arange(len(models))
        width  = 0.22
        fig, ax = plt.subplots(figsize=(12, 5))

        for i, (cat, label, color) in enumerate(zip(categories, cat_labels, cat_colors)):
            scores = [e.get("scores_by_category", {}).get(cat, 0) for e in leaderboard]
            ax.bar(x + i * width, scores, width, label=label, color=color, alpha=0.85, zorder=2)

        # Overall score line
        overall_scores = [e["overall_score"] for e in leaderboard]
        ax.plot(x + width, overall_scores, "ko--", linewidth=1.5,
                markersize=7, label="Overall", zorder=3)
        for xi, sc in zip(x + width, overall_scores):
            ax.text(xi, sc + 0.015, f"{sc:.2f}", ha="center", fontsize=9, fontweight="bold")

        ax.set_xticks(x + width)
        ax.set_xticklabels(models, fontsize=11)
        ax.set_ylim(0, 1.1)
        ax.set_ylabel("Score")
        ax.set_title(f"Model Benchmark Leaderboard — {bench_data.get('run_date')}",
                     fontsize=13, fontweight="bold")
        ax.legend()
        ax.grid(axis="y", alpha=0.3, zorder=1)
        ax.set_facecolor("#f9f9f9")

        plt.tight_layout()
        plt.savefig("model_benchmark.png", dpi=120, bbox_inches="tight")
        plt.show()
        print("📊 Chart saved to model_benchmark.png")

## 8. Agent Traces — Observability

Every agent query writes a trace to **OpenSearch** (`agent-traces-YYYY.MM.DD`). 
We read the last 7 days and visualize:
- Latency distribution
- Route breakdown (rag / data / both)

In [ ]:
from opensearchpy import OpenSearch

os_client = OpenSearch(
    hosts=[{"host": "opensearch", "port": 9200}],
    http_compress=True,
    use_ssl=False,
)

# ── Query last 7 days of agent traces ────────────────────────────────────────
since = (datetime.utcnow() - timedelta(days=7)).strftime("%Y-%m-%dT%H:%M:%SZ")
query = {
    "size": 500,
    "query": {"range": {"@timestamp": {"gte": since}}},
    "_source": ["@timestamp", "latency_ms", "route", "tool_used", "model", "chunks_retrieved"],
}

try:
    resp = os_client.search(index="agent-traces-*", body=query)
    hits = resp["hits"]["hits"]
    traces = [h["_source"] for h in hits]
    print(f"✅ Retrieved {len(traces)} agent traces (last 7 days)")
except Exception as e:
    traces = []
    print(f"⚠️  Could not connect to OpenSearch: {e}")
    print("   Make sure the notebook is running inside the cluster overlay network,")
    print("   or update OPENSEARCH_URL to the external URL.")

In [ ]:
# ── Visualize traces ──────────────────────────────────────────────────────────
if traces:
    df = pd.DataFrame(traces)
    df["latency_ms"] = pd.to_numeric(df["latency_ms"], errors="coerce")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle("Agent Traces — Last 7 Days", fontsize=13, fontweight="bold")

    # ── Latency histogram ─────────────────────────────────────────────────────
    ax = axes[0]
    df["latency_ms"].dropna().plot.hist(bins=20, ax=ax, color="#3498db", alpha=0.8, zorder=2)
    mean_lat = df["latency_ms"].mean()
    ax.axvline(mean_lat, color="red", linestyle="--",
               label=f"mean: {mean_lat:.0f} ms")
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("Count")
    ax.set_title("Response Latency Distribution")
    ax.legend()
    ax.grid(axis="y", alpha=0.3, zorder=1)
    ax.set_facecolor("#f9f9f9")

    # ── Route pie chart ───────────────────────────────────────────────────────
    ax2 = axes[1]
    route_counts = df["route"].value_counts()
    route_colors = {"rag": "#2ecc71", "data": "#3498db", "both": "#9b59b6"}
    colors_ordered = [route_colors.get(r, "#95a5a6") for r in route_counts.index]
    ax2.pie(route_counts.values, labels=route_counts.index,
            colors=colors_ordered, autopct="%1.1f%%",
            startangle=90, pctdistance=0.75)
    ax2.set_title("Route Distribution")

    plt.tight_layout()
    plt.savefig("agent_traces.png", dpi=120, bbox_inches="tight")
    plt.show()

    print(f"\n📊 Summary:")
    print(f"   Total traces : {len(df)}")
    print(f"   Mean latency : {mean_lat:.0f} ms")
    print(f"   p95 latency  : {df['latency_ms'].quantile(0.95):.0f} ms")
    print(f"   Route breakdown: {dict(route_counts)}")
else:
    print("No traces to visualize — run sections 4 and 5 first, then re-run this cell.")

---

## Summary

| Component | Status | Details |
|-----------|--------|---------|
| RAG API — Ingest | ✅ | Document chunked, embedded (bge-m3 1024d), stored in Qdrant + PostgreSQL |
| RAG API — Query | ✅ | Semantic search → Ollama LLM → answer with sources + scores |
| Agent — RAG route | ✅ | LangGraph routed to Qdrant, retrieved context, synthesized answer |
| Agent — Data route | ✅ | LangGraph routed to SQL, gemma4:26b generated query, PostgreSQL executed |
| RAGAS Evaluation | ✅ | Faithfulness · Answer Relevancy · Context Precision scored via LLM-as-judge |
| Model Benchmark | ✅ | Leaderboard across instruction following / reasoning / coding categories |
| Agent Observability | ✅ | Latency distribution + route breakdown from OpenSearch traces |

**Infrastructure**: 2-node bare-metal Docker Swarm · RTX 2080 Ti · 100% local inference · no cloud required

> See [`docs/adrs/`](../docs/adrs/) for architecture decisions and [`docs/ROADMAP.md`](../docs/ROADMAP.md) for the full project history.